# 00 Preprocessing: partition, cleaning and common cache

Runs once on CPU. Input: the Kaggle dataset *data de deforestación*
(https://www.kaggle.com/datasets/vergaramarcoseduardo/data-de-deforestacin).
Output (`/kaggle/working`): `X224.npy`, `meta.csv`, `stats.json`, `prep_params.json` and `calidad.csv`.
Publish the output as a private Kaggle dataset (named `arboles-cache-224` in this project) and add it as input of the other notebooks.

In [ ]:
# Cell 1: load the metadata
import os, json, glob, numpy as np, pandas as pd, torch, torch.nn.functional as F
from sklearn.model_selection import train_test_split
BASE = "/kaggle/input/datasets/vergaramarcoseduardo/data-de-deforestacin/data"
if not os.path.exists(f"{BASE}/patches_meta.csv"):                 # Kaggle may mount the dataset in another folder
    BASE = os.path.dirname(glob.glob("/kaggle/input/**/patches_meta.csv", recursive=True)[0])
OUT, SIZE = "/kaggle/working", 224
meta = pd.read_csv(f"{BASE}/patches_meta.csv")
ids = meta.patch_id.tolist()
print(len(ids), "patches")

In [ ]:
# Cell 2: stratified partition 80/10/10 (seed 42), before any statistic or augmentation
meta["split_original"] = meta["split"]                              # official partition of the dataset (1000/200/300), kept for reference
tr_ids, tmp = train_test_split(meta.patch_id, test_size=0.20, stratify=meta.label, random_state=42)
lab_tmp = meta.set_index("patch_id").loc[tmp, "label"]
va_ids, te_ids = train_test_split(tmp, test_size=0.50, stratify=lab_tmp, random_state=42)
meta["split"] = "train"
meta.loc[meta.patch_id.isin(va_ids), "split"] = "val"
meta.loc[meta.patch_id.isin(te_ids), "split"] = "test"
print(pd.crosstab(meta.label, meta.split, margins=True))            # 1200 / 150 / 150

In [ ]:
# Cell 3: cleaning parameters computed ONLY with training patches (200 patches, 50 per class)
rng = np.random.default_rng(42)
muestra = meta[meta.split == "train"].groupby("label").sample(50, random_state=42).patch_id
vals = [[] for _ in range(6)]
for pid in muestra:
    x = np.load(f"{BASE}/images/{pid}.npy")
    valido = ~(x == 0).all(axis=1)                                   # (5, 512, 512): False where the six bands are 0 (masked clouds)
    for b in range(6):
        v = x[:, b][valido]
        vals[b].append(rng.choice(v, min(20000, v.size), replace=False))
P = {"fill": [float(np.concatenate(v).mean()) for v in vals],
     "clip": [float(np.percentile(np.concatenate(v), 99.5)) for v in vals]}
json.dump(P, open(f"{OUT}/prep_params.json", "w"), indent=1); print(P)

In [ ]:
# Cell 4: cache (fill gaps, clip saturated values, resize the complete patch to 224, add NDVI and NBR)
fill = np.array(P["fill"], np.float32).reshape(1, 6, 1, 1)
clip = np.array(P["clip"], np.float32).reshape(1, 6, 1, 1)
X = np.lib.format.open_memmap(f"{OUT}/X224.npy", mode="w+", dtype=np.float16, shape=(len(ids), 5, 8, SIZE, SIZE))
calidad = []
for i, pid in enumerate(ids):
    x = np.load(f"{BASE}/images/{pid}.npy")
    vac = (x == 0).all(axis=1, keepdims=True)                        # (5, 1, 512, 512)
    calidad.append({"patch_id": pid, **{f"vacio_{2020+t}": float(vac[t].mean()) for t in range(5)}})
    x = np.minimum(np.where(vac, fill, x), clip)
    t = F.interpolate(torch.from_numpy(x), size=(SIZE, SIZE), mode="area")
    b4, b8, b12 = t[:, 2], t[:, 3], t[:, 5]
    ndvi = ((b8 - b4) / (b8 + b4 + 1e-6)).clamp(-1, 1)
    nbr = ((b8 - b12) / (b8 + b12 + 1e-6)).clamp(-1, 1)
    X[i] = torch.cat([t, ndvi[:, None], nbr[:, None]], 1).numpy().astype(np.float16)
    if i % 100 == 0: print(i, pid)
X.flush()
meta.to_csv(f"{OUT}/meta.csv", index=False)
pd.DataFrame(calidad).merge(meta[["patch_id", "label", "split"]]).to_csv(f"{OUT}/calidad.csv", index=False)

In [ ]:
# Cell 5: per-channel normalization computed ONLY with training patches
tr = np.where(meta.split == "train")[0]
s, ss, n = np.zeros(8), np.zeros(8), 0
for i in tr:
    x = X[i].astype(np.float64)
    s += x.sum((0, 2, 3)); ss += (x ** 2).sum((0, 2, 3))
    n += x.shape[0] * x.shape[2] * x.shape[3]
mean = s / n; std = np.sqrt(ss / n - mean ** 2)
json.dump({"mean": mean.tolist(), "std": std.tolist()}, open(f"{OUT}/stats.json", "w"), indent=1)
print("mean:", mean.round(3)); print("std:", std.round(3))

In [ ]:
# Cell 6: check the cache
Xc = np.load(f"{OUT}/X224.npy", mmap_mode="r")
print("shape:", Xc.shape)                                            # (1500, 5, 8, 224, 224)
print("finite:", all(np.isfinite(Xc[i].astype(np.float32)).all() for i in range(0, 1500, 50)))
print(pd.read_csv(f"{OUT}/meta.csv").split.value_counts())